# Lesson 41: No-code / low-code AI — n8n (email classifier) and Langflow 🧩🖱️
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Explain **workflow automation** and where AI fits in it
2. Understand **n8n** concepts: triggers, nodes, connections, credentials, executions
3. Plan the **automated email classifier** project in n8n — and build the same logic in Python to understand every step
4. Understand **Langflow**: visual building of LangChain-style apps, and calling a flow through its API

**⏱️ Timing:** automation & n8n concepts 10' | email classifier in Python 25' | n8n build (teacher demo) 10' | Langflow 15'

> 🧑‍🏫 **Teacher note:** n8n and Langflow are visual apps that run in a browser, not inside Colab. Students run the Python version below; the teacher demonstrates the visual version (n8n cloud trial, or self-hosted with Docker). Account sign-ups for these services are generally for adults, so the teacher should own the accounts.

> ⚙️ **GPU recommended:** Runtime → Change runtime type → **T4 GPU** (the local LLM runs much faster).

## 🏭 An analogy: a factory conveyor belt
**Workflow automation** connects apps so work flows by itself: *when* something happens (an email arrives), *do* a series of steps (read it, decide what it is, file it, notify someone).
- **Trigger** = what starts the belt (new email, a schedule, a form, a webhook)
- **Nodes** = the machines along the belt (Gmail, an AI model, an IF/Switch, Slack, Google Sheets…)
- **Connections** = the belt between machines; data flows as **items** (JSON)

**n8n** (pronounced "n-eight-n") is a popular open-source ("fair-code") automation tool with hundreds of app connectors plus AI nodes (LLM chains, agents, text classifiers). **Langflow** is a visual builder focused specifically on **LLM apps** (RAG, agents) with LangChain-style components.

| | n8n | Langflow | Python (this course) |
|---|---|---|---|
| Best for | Connecting apps + AI steps | Prototyping LLM apps visually | Full control |
| How you build | Drag nodes on a canvas | Drag components on a canvas | Write code |
| Runs as | Web app (cloud or self-hosted) | Web app (desktop, Docker or cloud) | Scripts, notebooks, servers |

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:3b"     # small and fast. For agent lessons, "qwen2.5:7b" is smarter (slower to download)
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install openai
from openai import OpenAI
import json, re
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

## Part 1: The email-classifier project in Python 📧
**Goal:** a school office gets lots of emails. Automatically sort each one into a category, extract key details, set a priority, and route it to the right person.

```
[New email] → [AI classifier] → [Switch by category] → ATTENDANCE → add to attendance sheet
                                                    → URGENT     → notify the principal now
                                                    → ENROLMENT  → forward to admissions
                                                    → OTHER      → label "General" in the inbox
```
We'll build each "node" as a Python function, so the n8n version later makes complete sense.

In [ ]:
# NODE 1 — Trigger: incoming emails (in n8n: the "Gmail Trigger" or "Email Trigger (IMAP)" node)
inbox = [
    {"from": "parent.kaur@example.com", "subject": "Arjun sick today", "body": "Hi, Arjun has a temperature and will stay home today. Thanks, Simran Kaur"},
    {"from": "j.smith@example.com", "subject": "Enrolment for 2027", "body": "We are moving to the area and would like to enrol our daughter in Year 9 next year. What do we need?"},
    {"from": "neighbour@example.com", "subject": "URGENT gate left open", "body": "The back gate on Hill Rd is broken and open, and young children are walking onto the road. Please send someone now."},
    {"from": "coach.lee@example.com", "subject": "Netball results", "body": "Great news: the senior netball team won 34-20 on Saturday! Could this go in the newsletter?"},
    {"from": "parent.wong@example.com", "subject": "Dentist", "body": "Please excuse Mei at 1:30pm tomorrow for a dentist appointment. She will return to school after."},
]
print(len(inbox), "emails waiting")

In [ ]:
# NODE 2 — AI classifier (in n8n: "Text Classifier" or "Basic LLM Chain" node + a chat-model sub-node)
CATEGORIES = ["ATTENDANCE", "ENROLMENT", "URGENT", "NEWSLETTER", "OTHER"]

def classify(email):
    prompt = f"""You sort emails for a school office. Read the email and reply with JSON only:
{{"category": one of {CATEGORIES},
  "priority": "high" | "normal" | "low",
  "student_name": the student's name or null,
  "summary": one short sentence}}

Subject: {email['subject']}
Body: {email['body']}"""
    r = client.chat.completions.create(model=MODEL, temperature=0, response_format={"type": "json_object"},
                                       messages=[{"role": "user", "content": prompt}])
    data = json.loads(r.choices[0].message.content)
    if data.get("category") not in CATEGORIES: data["category"] = "OTHER"     # guardrail: never trust the format blindly
    return data

result = classify(inbox[0])
print(json.dumps(result, indent=2))

In [ ]:
# NODE 3 — Switch: route by category (in n8n: the "Switch" node)
# NODES 4+ — Actions (in n8n: Google Sheets, Gmail "Add label", Slack/Telegram, "Send email"…)
attendance_sheet, admissions_queue, newsletter_drafts, alerts, general_label = [], [], [], [], []

def route(email, info):
    category = info["category"]
    if category == "ATTENDANCE":
        attendance_sheet.append({"student": info.get("student_name"), "note": info["summary"], "from": email["from"]})
    elif category == "ENROLMENT":
        admissions_queue.append(email["from"])
    elif category == "URGENT":
        alerts.append(f"🚨 To principal: {info['summary']}")
    elif category == "NEWSLETTER":
        newsletter_drafts.append(email["body"])
    else:
        general_label.append(email["subject"])
    return category

# The workflow "execution": run every email through the nodes, and keep a log (n8n shows this as the Executions list)
log = []
for email in inbox:
    info = classify(email)
    category = route(email, info)
    log.append({"subject": email["subject"], "category": category, "priority": info.get("priority")})

import pandas as pd
display(pd.DataFrame(log))
print("📋 Attendance sheet:", attendance_sheet)
print("🚨 Alerts:", alerts)
print("🎓 Admissions queue:", admissions_queue)

### 🧩 Exercise 1
1. Write 5 tricky emails (two topics in one email, sarcasm, a different language, a very short one). Which get misclassified?
2. **Human in the loop:** for `URGENT`, would you let the AI send a message automatically, or ask a person first? What about replying to parents?
3. **Privacy:** these emails include children's names and health information. What rules should a school follow before sending emails to an AI service? (Consent, data agreements, using a local model like ours, deleting logs…)

## Part 2: Building it in n8n (teacher demo) 🖱️
**Run n8n:** use the n8n cloud trial, or self-host with Docker on a computer (not Colab): `docker run -it --rm -p 5678:5678 n8nio/n8n` → open http://localhost:5678

| # | n8n node | Settings | Python equivalent |
|---|---|---|---|
| 1 | **Gmail Trigger** (or Email Trigger IMAP) | Poll every minute; connect a test inbox via **Credentials** | `inbox` list |
| 2 | **Text Classifier** (AI) | Categories: Attendance, Enrolment, Urgent, Newsletter, Other — each with a description | `classify()` |
| 2a | ↳ **Chat Model** sub-node | OpenAI / Anthropic / Gemini with a key, or **Ollama** for a local model | `client.chat.completions…` |
| 3 | One output branch per category (Text Classifier has one output per category; or use a **Switch** node) | — | `route()` |
| 4 | **Google Sheets → Append row** | Attendance branch | `attendance_sheet.append` |
| 5 | **Gmail → Add label** | Other branch | `general_label.append` |
| 6 | **Slack / Telegram / Send Email** | Urgent branch, to the principal | `alerts.append` |

Then click **Test workflow**, send a test email, and open **Executions** to see the data at every node (like our `log`). Turn it **Active** to run it automatically.

> 💡 n8n's node names and options change between versions — follow the current n8n docs (docs.n8n.io) and its built-in templates (search "email classification").

### 🧩 Exercise 2
Draw your own n8n workflow on paper for another school task: e.g. "When a Google Form for sports sign-ups is submitted → AI checks the answers → add to sheet → email the coach." Label the trigger, each node and what data passes between them.

## Part 3: Langflow — visual LLM apps 🌊
**Langflow** lets you drag **components** (inputs, prompts, models, vector stores, agents, outputs) and connect them to build an LLM app, then call it as an **API**.
- **Install** on a computer: `pip install langflow` then `langflow run` (or use Docker / Langflow's cloud) → open http://localhost:7860
- Start from a template such as **Basic Prompting**, **Vector Store RAG** or **Simple Agent**
- The **Playground** lets you chat with your flow; the **API** button shows code to call it

A simple RAG flow in Langflow looks like this:
```
[Chat Input] ──► [Vector Store (search)] ──► [Parser] ──┐
                         ▲                              ├──► [Prompt] ──► [Ollama/OpenAI model] ──► [Chat Output]
[File] ─► [Split Text] ─► [Embeddings] ─► (store)       │
                              [Chat Input] ─────────────┘
```
Every visual component maps to code you already know:

In [ ]:
# The same "Basic Prompting" flow as code: Chat Input → Prompt → Model → Chat Output
def prompt_component(question, persona="a friendly science tutor for 14-year-olds"):
    return f"You are {persona}. Answer in 3 sentences.\n\nQuestion: {question}"

def model_component(prompt):
    return client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}]).choices[0].message.content

def chat_output(text):
    print("🤖", text)

chat_output(model_component(prompt_component("Why does the Moon have phases?")))

In [ ]:
# Calling a Langflow flow from Python (works when Langflow is running and you have a flow ID).
# The API button in Langflow shows the exact URL, ID and headers for YOUR flow.
import requests
LANGFLOW_URL = "http://localhost:7860"
FLOW_ID = "paste-your-flow-id-here"

def run_flow(message):
    r = requests.post(f"{LANGFLOW_URL}/api/v1/run/{FLOW_ID}",
                      json={"input_value": message, "input_type": "chat", "output_type": "chat"}, timeout=60)
    r.raise_for_status()
    return r.json()

if FLOW_ID != "paste-your-flow-id-here":
    print(run_flow("Why does the Moon have phases?"))
else:
    print("ℹ️ Langflow isn't running in Colab — this cell shows how an app would call a deployed flow.")

### 🧩 Exercise 3: Visual vs code
| Question | n8n | Langflow | Python |
|---|---|---|---|
| Fastest to build a first version? | | | |
| Easiest for a non-programmer to change? | | | |
| Easiest to test and track with version control (Git)? | | | |
| Best for connecting Gmail, Sheets and Slack? | | | |
| Most control over every detail? | | | |

### 🚀 Challenge
Extend the Python email classifier: add a **"draft reply"** node that writes a polite reply for `ENROLMENT` emails, but **never sends it** — it saves it for a human to review.

## 📌 Summary
- **Workflow automation**: trigger → nodes → actions, with data passed between them
- **n8n** connects hundreds of apps with AI nodes — great for automations like the **email classifier**
- **Langflow** builds LLM apps (RAG, agents) visually and exposes them as APIs
- Visual tools are fast and friendly; code gives control, testing and version history. Keep **privacy** and **human review** in mind